# Plotting a district heating network on an interactive map

This tutorial shows how to draw a pandapipes network on a web map with `create_map_figure` and how to read the temperatures in a **closed heating loop**. We use the district heating grid of the town *Schutterwald* that ships with pandapipes: a central circulation pump heats the water to 70 °C, the **supply** pipes carry it to the consumers, and the **return** pipes bring the cooled water back.

The map uses plotly's MapLibre support, so **no access token or account is needed**. Plotly >= 5.24 is required (`pip install "pandapipes[plotting]"`). The background tiles are loaded by your browser from public tile servers, so you need an internet connection to see the street map.

In [ ]:
import pandas as pd

import pandapipes as pp
import pandapipes.networks as nw
from pandapipes.plotting import (create_map_figure, convert_epsg_junction_geodata,
                                 convert_epsg_pipe_geodata)

## Preparing the coordinates

`create_map_figure` expects WGS84 longitude (`x`) and latitude (`y`). The example network uses a projected coordinate system (Gauss-Krüger zone 3, EPSG:31467), and it only stores coordinates for the supply side. We therefore

1. place every return junction about 6 m next to its supply partner (return junctions are named `return_<supply name>`), so supply and return lines can be told apart on the map, and
2. convert junction and pipe coordinates to longitude/latitude.

For your own network that already has longitude/latitude coordinates, skip this step.

In [ ]:
net = nw.schutterwald_heat()

names = net.junction.name
has_geo = net.junction_geodata.x.notna()
supply_of = pd.Series(names[has_geo].index, index=names[has_geo])
return_junctions = names.index[~has_geo]
partners = supply_of[names[return_junctions].str.removeprefix("return_")].values

shift = [5.0, -4.0]  # metres
net.junction_geodata.loc[return_junctions, ["x", "y"]] = \
    net.junction_geodata.loc[partners, ["x", "y"]].values + shift
is_return_pipe = net.pipe.from_junction.isin(return_junctions)
net.pipe_geodata.loc[is_return_pipe, "coords"] = net.pipe_geodata.loc[is_return_pipe, "coords"].apply(
    lambda coords: [[x + shift[0], y + shift[1]] for x, y in coords])

convert_epsg_junction_geodata(net, epsg_in=31467, epsg_out=4326)
convert_epsg_pipe_geodata(net, epsg_in=31467, epsg_out=4326)

## Network overview

Before a calculation the figure shows the topology: pipes, junctions and components such as the circulation pump, heat consumers and valves. Hover over an element to see its index.

In [ ]:
fig = create_map_figure(net)
fig.show()

## Temperatures in the heating loop

We run a pipeflow with `mode="sequential"` (hydraulics, then temperatures). Any column of `net.res_pipe` can be used to color the pipes. Plotly shows the values as they are, so we first add the temperature in °C.

In [ ]:
pp.pipeflow(net, mode="sequential", iter=100)
net.res_pipe["t_to_c"] = net.res_pipe["t_to_k"] - 273.15

fig = create_map_figure(net, pipe_result="t_to_c", colorbar_title="Water temperature [°C]")
fig.show()

The hot water (red) leaves the central pump and cools down on its way through the supply pipes because of heat losses to the surroundings. The return pipes (blue to purple) carry the water that has given off its heat at the consumers. **Pipes without flow** are drawn in grey and are left out of the color range; their temperature is just the ambient temperature and would otherwise distort the colors.

Let us check the numbers behind the picture:

In [ ]:
res = net.res_pipe
flow = res.mdot_from_kg_per_s.abs() > 1e-9
summary = pd.DataFrame({
    "supply pipes": res.t_to_c[flow & ~is_return_pipe].agg(["min", "mean", "max"]),
    "return pipes": res.t_to_c[flow & is_return_pipe].agg(["min", "mean", "max"]),
}).round(1)
summary.index.name = "temperature [°C]"
summary

In [ ]:
pump = net.res_circ_pump_pressure.iloc[0]
print(f"Circulation pump: water returns at {pump.t_from_k - 273.15:.1f} °C and is heated to "
      f"{pump.t_to_k - 273.15:.1f} °C at {pump.mdot_from_kg_per_s:.1f} kg/s "
      f"({pump.qext_w / 1e6:.2f} MW heat input)")

Supply temperatures fall from 70 °C at the plant to roughly 49 °C at the far ends of the grid, while the water returning to the plant is about 52 °C on average. This is the typical picture of a closed heating loop: the temperature spread between supply and return is what delivers the heat to the consumers.

## Other map styles and sharing

Choose another token-free basemap with `map_style`, e.g. `"open-street-map"` or `"carto-darkmatter"`. Use `fig.write_html("network.html")` to share the interactive map as a standalone file.

In [ ]:
fig = create_map_figure(net, pipe_result="t_to_c", colorbar_title="Water temperature [°C]",
                         map_style="open-street-map")
fig.show()